In [ ]:
import os
import datetime
import ollama
import stanza


In [ ]:
import sys
import torch
from importlib import reload

# Intercept and patch the global torch load function
original_load = torch.load
def patched_load(*args, **kwargs):
    kwargs['weights_only'] = False
    return original_load(*args, **kwargs)
torch.load = patched_load

# Reload stanza if it has already been initialized in this session
if 'stanza' in sys.modules:
    reload(sys.modules['stanza'])
    
import numpy as np

# Tell PyTorch's unpickler that NumPy's multiarray reconstructor is safe to load
torch.serialization.add_safe_globals([
    np.core.multiarray._reconstruct,
    np.dtype
])

import spacy_stanza
import random
stanza.download("he")

import json
from jsonschema import validate, ValidationError

In [ ]:
def save_ai_story(prompt, filepath, model_name='llama3.1'):
    dir_name = os.path.dirname(filepath)
    if dir_name and not os.path.exists(dir_name):
        os.makedirs(dir_name)
        print(f"Created missing directory structure: {dir_name}")

    print(f"Querying Ollama ({model_name}) for: {os.path.basename(filepath)}...")
    
    # Pass the targeted model_name dynamically here
    response = ollama.generate(model=model_name, prompt=prompt)
    story_text = response['response']
    
    with open(filepath, 'w', encoding='utf-8') as file:
        file.write(story_text)
        
    print(f"Successfully generated and saved story to: {filepath}")
    return story_text

# Story Generator

Generates a 200-300 word story in English and Hebrew to be used for the quests and quizzes.

## Steps

1. English Narrative Generation via Llama
1. Hebrew Translation Optimization via DictaLM

In [ ]:
# Core story elements
level = "CEFR A1/A2"
character_name = "Faun"
animal_type = "faun"
gender = "Male"
quest_name = "garden_adventure"

# Story guidance variables
story_title = "יום הגינון של פאון (Faun's Gardening Day)"
story_overview = (
    "Faun has a strong רָעָב (hunger) for a sweet, crunchy treat. "
    "He decides to become a gardener for a day, taking a סַל (basket) "
    "of tools out to the sunny גַּן (garden). He carefully buries a small "
    "זֶרַע (seed) in the dirt, waters it, and proudly harvests his very "
    "first home-grown גֶּזֶר (carrot)."
)

# Python list of vocabulary words
word_list = ["רעב", "סל", "גן", "זרע", "גזר"]

# Format the word list for the prompt
formatted_words = "\n".join([f"- {word}" for word in word_list])

# English Generation Prompt
large_prompt_en = f"""
You are an expert children's book author. Write an original, charming short story in simple, clear English based on this plot overview: 
"{story_overview}"

Rules:
- Keep the sentences short and clear so they translate cleanly into early-intermediate language structures.
- Do not create any other names, use nouns only (bird, girl, city, etc).
- Do not explain grammar or add any extra text to the story, aside from the story text itself.
- Do not format any text in the story aside from using new lines.
"""

# Generate the unique timestamped filenames to prevent overriding
timestamp = datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
animal_folder = animal_type.lower().replace(" ", "_")
filename = f"{timestamp}_{quest_name}"

# Generate the English story using llama3.1
english_story_output = save_ai_story(
    prompt=large_prompt_en,
    filepath=f"./quests/{animal_folder}/{filename}_en.md",
    model_name='llama3.1'
)

# Construct the dynamic Hebrew translation prompt, feeding it the English text
large_prompt_he = f"""
You are an expert bilingual translator fluent in both English and Hebrew. 

Translate the following English story into grammatically correct, natural Hebrew suitable for a CEFR A1/A2 language learner. 

English Story to Translate:
\"\"\"
{english_story_output}
\"\"\"

Rules for the translation:
- Do NOT use vowel points (nikkud) at all. Write in clean, modern, unpointed Hebrew text (Ktav Male).
- Ensure strict gender agreement (since the main character {character_name} is {gender}, use proper masculine verb inflections and adjectives).
- Match the key concepts exactly to these Hebrew vocabulary items:
{formatted_words}
- Do not explain grammar, do not add conversational notes, and do not include the original English in your output. Return ONLY the Hebrew translation.

CRITICAL OUTPUT FORMATTING RULES:
- Provide ONLY the direct Hebrew translation.
- DO NOT include introductory remarks like "Here is the translation...".
- DO NOT include conversational text, pleasantries, or explanations.
- Start your response directly with the translated Hebrew title or the first line of the translated story text.
- Match the new lines from the English version.
"""

# Pass the dynamic translation prompt specifically to DictaLM, which has been built specifically for Hebrew
hebrew_story_output = save_ai_story(
    prompt=large_prompt_he,
    filepath=f"./quests/{animal_folder}/{filename}_he.md",
    model_name='aminadaven/dictalm2.0-instruct:q4_k_m'
);

## Generate Quiz

In [ ]:
# 2. Initialize the Stanza Hebrew pipeline wrapper
nlp = spacy_stanza.load_pipeline("he")

def extract_cloze_quizzes(story_hebrew, target_words):
    """
    Parses Hebrew text, masks target words, and uses Part-of-Speech (POS) tags
    to generate grammatically matching distractor choices for multiple-choice questions.
    """
    doc = nlp(story_hebrew)
    cloze_quizzes = list()  # Initialized as a list to store generated items
    
    # Catalog all words in the story by their POS tags
    pos_bank = dict()
    for token in doc:
        if token.is_alpha and not token.is_stop:
            pos = token.pos_  # Universal POS tags (e.g., 'NOUN', 'VERB', 'ADJ')
            # Store lemmas (base forms) to prevent duplicate inflections
            pos_bank.setdefault(pos, set()).add(token.lemma_)
            
    # Iterate through sentences to find and mask target words
    for sentence in doc.sents:
        sentence_text = sentence.text
        
        for target in target_words:
            # Locate target token based on exact text or base lemma
            target_token = next(
                (t for t in sentence if t.text == target or t.lemma_ == target), 
                None
            )
            
            if target_token:
                # Replace the exact word instance in the sentence with a blank card
                blank_text = sentence_text.replace(target_token.text, "_____")
                
                # Fetch words from the story that share the exact same POS tag
                target_pos = target_token.pos_
                distractor_pool = list(pos_bank.get(target_pos, set()))
                
                # Remove the correct answer from the distractors list
                distractor_pool = [d for d in distractor_pool if d!= target_token.lemma_ and d!= target_token.text and d!= target]
                
                # Fallbacks in case the story is too short to provide 3 matching POS options
                fallbacks = ["בית", "חבר", "מקום", "ילד"]
                while len(distractor_pool) < 3:
                    distractor_pool.append(random.choice(fallbacks))
                
                # Select 3 distractors, combine with correct answer, and shuffle
                distractors = random.sample(distractor_pool, 3)
                options = distractors + [target_token.text]
                random.shuffle(options)
                
                cloze_quizzes.append({
                    "type": "cloze",
                    "prompt": {
                        "male": blank_text,
                        "female": blank_text,
                        "neutral": blank_text
                    },
                    "answer": target_token.text,
                    "options": options
                })
                break # Limit to one blank per sentence to balance cognitive load
                
    return cloze_quizzes

In [ ]:


# 1. Define the distantlife quest schema to catch errors before deployment
QUEST_SCHEMA = {
    "type": "object",
    "properties": {
        "quest_id": {"type": "string"},
        "locale": {"type": "string"},
        "theme": {"type": "string"},
        "quest_type": {"type": "string"},
        "quest_line_id": {"type": "string"},
        "allowed_pet_type_ids": {
            "type": "array",
            "items": {"type": "integer"}
        },
        "unlock_cost": {"type": "integer"},
        "review_status": {"type": "string"},
        "episodes": {
            "type": "array",
            "items": {
                "type": "object",
                "properties": {
                    "episode_id": {"type": "string"},
                    "title": {"type": "string"},
                    "story_text": {
                        "type": "object",
                        "properties": {
                            "male": {"type": "string"},
                            "female": {"type": "string"},
                            "neutral": {"type": "string"}
                        },
                        "required": ["male", "female", "neutral"]
                    },
                    "speech_bubble_lines": {
                        "type": "array",
                        "items": {
                            "type": "object",
                            "properties": {
                                "male": {"type": "string"},
                                "female": {"type": "string"},
                                "neutral": {"type": "string"}
                            }
                        }
                    },
                    "quiz": {
                        "type": "object",
                        "properties": {
                            "questions": {"type": "array"}
                        },
                        "required": ["questions"]
                    }
                },
                "required": ["episode_id", "title", "story_text", "quiz"]
            }
        }
    },
    "required": ["quest_id", "locale", "theme", "quest_type", "quest_line_id", "allowed_pet_type_ids", "episodes"]
}

In [ ]:
# Raw inputs from your Ollama story generation step
story_hebrew_raw = hebrew_story_output
story_english_raw = english_story_output
vocab_targets = word_list

# Process the quizzes
quizzes = extract_cloze_quizzes(story_hebrew_raw, vocab_targets)

# Package into the canonical distantlife quest structure
quest_payload = {
    "quest_id": "hungry_carrot_01",
    "locale": "he",
    "theme": "vegetables",
    "quest_type": "story_quest",
    "quest_line_id": "hungry_carrot",
    "allowed_pet_type_ids": [1],  # Restricted specifically to Faun (ID 10)
    "unlock_cost": 50,
    "review_status": "approved",
    "episodes": [
        {
            "episode_id": "hungry_carrot_01_ep1",
            "title": "הסל הריק",
            "story_text": {
                "male": "הפאון התעורר רעב. הוא בדק את הסל שלו, אבל הוא היה ריק. הוא הלך לגן הכפר וראה אדמה יבשה. גנן נתן לו זרע של גזר.",
                "female": "הפאון התעוררה רעבה. היא בדקה את הסל שלה, אבל הוא היה ריק. היא הלכה לגן הכפר וראתה אדמה יבשה. גנן נתן לה זרע של גזר.",
                "neutral": "הפאון התעורר רעב. הוא בדק את הסל שלו, אבל הוא היה ריק. הוא הלך לגן הכפר וראה אדמה יבשה. גנן נתן לו זרע של גזר."
            },
            "speech_bubble_lines": [
                {
                    "male": "אני ממש רעב.",
                    "female": "אני ממש רעבה.",
                    "neutral": "אני ממש רעב."
                },
                {
                    "male": "הסל שלי ריק.",
                    "female": "הסל שלי ריק.",
                    "neutral": "הסל שלי ריק."
                },
                {
                    "male": "אני אשתול את הזרע הזה.",
                    "female": "אני אשתול את הזרע הזה.",
                    "neutral": "אני אשתול את הזרע הזה."
                }
            ],
            "quiz": {
                "questions": quizzes
            }
        }
    ]
}

# Validate and save
try:
    validate(instance=quest_payload, schema=QUEST_SCHEMA)
    print("JSON validation passed! Enforcing schema alignment.")
    
    # Save directly to your Flask app's static files directory
    output_path = "./quests/hungry_carrot_01.json"
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    
    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(quest_payload, f, ensure_ascii=False, indent=2)
        
    print(f"Success! Quest file written directly to assets folder: {output_path}")
    
except ValidationError as e:
    print(f"Schema violation detected: {e.message}")